In [ ]:
!pip install streamlit -q
!pip install pyngrok -q
!npm install -g localtunnel -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 16.7 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [ ]:
from google.colab import drive
import urllib

# 1. Kết nối với Drive
drive.mount('/content/drive')
# Ví dụ: '/content/drive/MyDrive/Project_Logistic_Regression/'
PATH_DRIVE = '/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/'

# 3. Copy các file cần thiết từ Drive vào thư mục /content/ của Colab
!cp "{PATH_DRIVE}model.keras" /content/
!cp "{PATH_DRIVE}logistic_stacking_artifacts.pkl" /content/
!cp "{PATH_DRIVE}wide_features.pkl" /content/
!cp "{PATH_DRIVE}categories_columns.pkl" /content/

Mounted at /content/drive


In [ ]:
# 2. Lấy địa chỉ IP Public (Dùng làm mật khẩu để vào App)
import urllib
print("Mật khẩu (Endpoint IP) để vào App là:", urllib.request.urlopen('https://ident.me').read().decode('utf8'))

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf

# --- 1. CẤU HÌNH & GIAO DIỆN ---
st.set_page_config(page_title="Hệ thống Quyết định Tín dụng", layout="wide")

def get_risk_bucket(probability):
    if probability <= 0.08: return "A", "Rất thấp", "🟢", "#d4edda"
    elif probability <= 0.15: return "B", "Thấp", "🔵", "#cce5ff"
    elif probability <= 0.30: return "C", "Trung bình", "🟡", "#fff3cd"
    elif probability <= 0.50: return "D", "Cao", "🟠", "#ffe5d0"
    else: return "E", "Rất cao", "🔴", "#f8d7da"

# --- 2. HÀM TIỀN XỬ LÝ (GIỮ NGUYÊN LOGIC FIX LỖI) ---
def deep_preprocess(df_input, all_columns, wide_features, label_encoders, impute_values):
    df = df_input.copy()
    for col in df.columns:
        clean_name = col.replace('_x', '').replace('_y', '')
        if clean_name != col and clean_name not in df.columns:
            df[clean_name] = df[col]
    for col, le in label_encoders.items():
        if col in df.columns:
            val = df[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else 0)
            df[f'{col}_index'] = val
            df[f'{col}_label'] = val
    cat_cols = [c for c in df.columns if c in label_encoders.keys()]
    df_encoded = pd.get_dummies(df, columns=cat_cols)
    full_needed_cols = list(set(all_columns) | set(wide_features))
    df_final = df_encoded.reindex(columns=full_needed_cols)
    for col, val in impute_values.items():
        if col in df_final.columns:
            df_final[col] = df_final[col].fillna(val)
    return df_final.fillna(0)

# --- 3. TẢI TÀI NGUYÊN ---
@st.cache_resource
def load_assets():
    model = tf.keras.models.load_model("/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/model.keras", compile=False)
    with open("/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/logistic_stacking_artifacts.pkl", "rb") as f:
        artifacts = pickle.load(f)
    with open("/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/all_columns.pkl", "rb") as f:
        all_columns = pickle.load(f)
    with open("/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/wide_features.pkl", "rb") as f:
        wide_features = list(pickle.load(f))
    with open("/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/label_encoders.pkl", "rb") as f:
        label_encoders = pickle.load(f)
    with open("/content/drive/MyDrive/Home_credit_risk_ARTIFACT/train/impute_values.pkl", "rb") as f:
        impute_values = pickle.load(f)
    return model, artifacts, all_columns, wide_features, label_encoders, impute_values

# --- 4. LOGIC CHÍNH ---
def main():
    st.title("Phân Tích & Quyết Định Tín Dụng Tự Động")

    try:
        model, artifacts, all_columns, wide_features, label_encoders, impute_values = load_assets()
        st.sidebar.success("Hệ thống đã sẵn sàng")
    except Exception as e:
        st.sidebar.error(f"Lỗi: {e}")
        st.stop()

    uploaded_file = st.file_uploader("Tải lên file CSV khách hàng", type="csv")

    if uploaded_file:
        df_raw = pd.read_csv(uploaded_file)
        if st.button("Chạy Phân Tích & Đề Xuất"):
            try:
                df_proc = deep_preprocess(df_raw, all_columns, wide_features, label_encoders, impute_values)
                X_wide = df_proc[wide_features]
                logits = []
                for item in artifacts:
                    scaled = item['scaler'].transform(X_wide)
                    logits.append(item['model'].decision_function(scaled))
                wide_in = np.mean(logits, axis=0).reshape(-1, 1).astype('float32')
                deep_in = np.zeros((len(df_raw), 12, 5)).astype('float32')
                preds = model.predict([wide_in, deep_in]).flatten()

                # Hiển thị kết quả tổng quan
                st.subheader("Kết quả phân hạng")

                results_list = []
                for i, p in enumerate(preds):
                    grade, desc, icon, color = get_risk_bucket(p)
                    results_list.append({
                        "SK_ID_CURR": df_raw.iloc[i]['SK_ID_CURR'],
                        "Xác suất vỡ nợ": f"{p:.2%}",
                        "Hạng": grade,
                        "Trạng thái": f"{icon} {desc}",
                        "color": color
                    })

                res_df = pd.DataFrame(results_list)
                st.dataframe(res_df.drop(columns=['color']).style.apply(lambda x: [f"background-color: {res_df.iloc[x.name]['color']}" for _ in x], axis=1))

                # --- PHẦN MỚI: GỢI Ý HÀNH ĐỘNG CHI TIẾT ---
                st.divider()
                st.subheader("Gợi ý hành động chi tiết cho từng hồ sơ")

                for i, p in enumerate(preds):
                    grade, desc, icon, color = get_risk_bucket(p)
                    customer_id = df_raw.iloc[i]['SK_ID_CURR']

                    with st.expander(f"Hồ sơ {customer_id} - Hạng {grade} {icon}"):
                        col_info, col_action = st.columns([1, 2])

                        with col_info:
                            st.write(f"**Xác suất rủi ro:** {p:.2%}")
                            st.write(f"**Phân nhóm:** {desc}")

                        with col_action:
                            if grade == "A":
                                st.success("**HÀNH ĐỘNG: PHÊ DUYỆT TỰ ĐỘNG (GREEN LANE)**")
                                st.write("- Chấp nhận giải ngân ngay lập tức.")
                                st.write("- Đề xuất áp dụng lãi suất ưu đãi thấp nhất.")
                                st.write("- Có thể mời chào thêm các sản phẩm thẻ tín dụng hạn mức cao.")
                            elif grade == "B":
                                st.info("**HÀNH ĐỘNG: PHÊ DUYỆT NHANH**")
                                st.write("- Chấp nhận giải ngân.")
                                st.write("- Áp dụng lãi suất tiêu chuẩn.")
                            elif grade == "C":
                                st.warning("**HÀNH ĐỘNG: THẨM ĐỊNH BỔ SUNG**")
                                st.write("- Yêu cầu bổ sung sao kê lương 6 tháng gần nhất.")
                                st.write("- Cần nhân viên thẩm định gọi điện kiểm tra nơi làm việc.")
                                st.write("- Xem xét giảm hạn mức vay dự kiến.")
                            elif grade == "D":
                                st.error("**HÀNH ĐỘNG: TỪ CHỐI HOẶC SIẾT CHẶT**")
                                st.write("- Ưu tiên từ chối để đảm bảo an toàn danh mục.")
                                st.write("- Nếu duyệt: Bắt buộc có tài sản đảm bảo hoặc người bảo lãnh uy tín.")
                                st.write("- Áp dụng mức phí rủi ro và lãi suất cao nhất.")
                            else: # Grade E
                                st.error("**HÀNH ĐỘNG: TỪ CHỐI NGAY LẬP TỨC (RED FLAG)**")
                                st.write("- Hồ sơ nằm trong nhóm nguy cơ vỡ nợ cực cao.")
                                st.write("- Không đề xuất các hình thức vay tín chấp.")
                                st.write("- Lưu lịch sử vào danh sách theo dõi đặc biệt.")

            except Exception as e:
                st.error(f"Lỗi: {e}")

if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
from pyngrok import ngrok

# Ngắt kết nối tất cả các tunnel hiện có
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    ngrok.disconnect(tunnel.public_url)

# Kill tiến trình ngrok đang chạy ngầm
import os
os.system("killall ngrok")

ERROR:pyngrok.process.ngrok:t=2026-01-15T07:40:55+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-01-15T07:40:55+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
from pyngrok import ngrok

# Thay YOUR_TOKEN_CỦA_BẠN bằng mã bạn vừa copy
NGROK_AUTH_TOKEN = "38E8Frz39YIJtNHrpXx12bZIKtA_25aRqAf5QY1ruXHUJABnP"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Chạy Streamlit và Ngrok
import os
os.system("nohup streamlit run app.py &")
public_url = ngrok.connect(8501).public_url
print(f"Truy cập App tại: {public_url}")

Truy cập App tại: https://beneficeless-hang-unverbally.ngrok-free.dev


In [ ]:
!cat app.py